# Paper B — Block E: second byte-level-BPE encoder (EuroBERT-210m)
Gate for the long paper. Same English detokenized WikiANN split, uniform LR 3e-5, O_WEIGHT 0.615, seeds 42/123/7, joint vs BIO — identical to Block B, so EuroBERT scores next to RoBERTa and XLM-R in `IdiomatorRigor_NER_en_bbpe`.

**Prediction (state before running):** EuroBERT uses the Llama-3 tokenizer (byte-level BPE, regex pre-tokenizer splits punctuation from words) → no SentencePiece artifact → QA−BIO gap ≈ 0 in original/extend/strip. If the gap is large, the "byte-BPE is immune" claim is dead and that is the finding.

**Needs the trainer patch pushed to GitHub first** (`TRUST_REMOTE_CODE` env + optional `token_type_ids`); cell 2 asserts it. Runtime → T4 or A100.

In [ ]:
# 1. Setup
from google.colab import drive
drive.mount('/content/drive')
%cd /content
!rm -rf /content/repo && git clone https://github.com/JustLetMeBeHello/Idiomator_Research.git /content/repo
%cd /content/repo
!git log --oneline -1
!pip install -q -r Requirements.txt datasets
!pip install -q "transformers>=4.48,<5"  # EuroBERT remote code needs ROPE_INIT_FUNCTIONS['default'] (removed in transformers 5)
import torch
assert torch.cuda.is_available(), 'CPU-only runtime — Runtime > Change runtime type > GPU, then remount + rerun'
print('Device: cuda —', torch.cuda.get_device_name(0))

In [ ]:
# 2. Preflight: patch is pushed, data exists on Drive, Block B roots writable
import os, json, glob, time
assert 'TRUST_REMOTE_CODE' in open('training/Train_Join.py').read() and 'TRUST_REMOTE_CODE' in open('experiments/ablations/BiO_Task_mBERT_train.py').read(), 'trainer patch not pushed to GitHub'
assert '"eurobert"' in open('experiments/rigor/run_08_extended_gold.py').read(), 'run_08 patch not pushed'
M = '/content/drive/MyDrive'
ROOT_B = f'{M}/IdiomatorRigor_NER_en_bbpe'
os.makedirs(ROOT_B, exist_ok=True)
open(f'{ROOT_B}/.probe','w').write('ok'); assert open(f'{ROOT_B}/.probe').read()=='ok'; os.remove(f'{ROOT_B}/.probe')
assert os.path.exists('data/ner_wikiann_multi/Splits/test.jsonl'), 'multilingual split missing from repo checkout'
print(sorted(os.listdir(ROOT_B)))

In [ ]:
# 3. Tokenizer check (no training): is EuroBERT's tokenizer really byte-level BPE, and does it give the comma its own word_id?
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained('EuroBERT/EuroBERT-210m', trust_remote_code=True)
print(type(tok).__name__, '| backend:', json.loads(tok.backend_tokenizer.to_str())['pre_tokenizer'] and json.loads(tok.backend_tokenizer.to_str())['pre_tokenizer'].get('type'))
s = 'The food, though, was great.'
enc = tok(s, return_offsets_mapping=True)
for t, o, w in zip(enc.tokens(), enc['offset_mapping'], enc.word_ids()): print(f'{t!r:14} {o} word_id={w}')
# Decision rule: if ',' shares word_id with 'food', EuroBERT is NOT immune — stop and report; do not run the block as a byte-BPE control.

In [ ]:
# 4. Dry-run (1 epoch, both trainers) — fails loudly if token_type_ids / remote-code handling is wrong
MODEL, ENC = 'answerdotai/ModernBERT-base', 'modernbert'  # EuroBERT-210m remote code breaks on transformers 4.57 and 5.x (rope 'default')
ENV_EN = "LANGS=English TEST_LANGS=English O_WEIGHT=0.615 LR_JOINT=3e-5 LR_BIO=3e-5 WORK_LOCAL=1 TRUST_REMOTE_CODE=1"
!DRIVE_OUT={ROOT_B} DATA_DIR=data/ner_wikiann_multi/Splits {ENV_EN} OUT_SUB=bbpe MODEL={MODEL} ENC_SHORT={ENC} bash experiments/rigor/run_07_mdeberta_sp_replication.sh --dry-run
!rm -rf {ROOT_B}/bbpe/_dryrun_joint {ROOT_B}/bbpe/_dryrun_bio

In [ ]:
# 5. Launch: EuroBERT-210m × seeds 42 123 7 × {joint, bio}. Resumable (metrics.json skip). 6 runs.
FORCE = 0
!DRIVE_OUT={ROOT_B} DATA_DIR=data/ner_wikiann_multi/Splits {ENV_EN} FORCE={FORCE} SEEDS="42 123 7" OUT_SUB=bbpe MODEL={MODEL} ENC_SHORT={ENC} bash experiments/rigor/run_07_mdeberta_sp_replication.sh

In [ ]:
# 6. Read back (500 rows each expected; a missing BIO row = skipped example) then score with RoBERTa + EuroBERT + (existing) XLM-R in one table
for f in sorted(glob.glob(f'{ROOT_B}/bbpe/{ENC}_*_s*/test_predictions.jsonl')):
    print(f'{sum(1 for _ in open(f)):5d} rows  {time.ctime(os.path.getmtime(f))}  {f[len(ROOT_B)+1:]}')
!BBPE=1 python experiments/rigor/run_08_extended_gold.py --preds-root {ROOT_B} --seeds 42 123 7 --out {ROOT_B}/run_08_ner_en_bbpe_{ENC}_3seed.json --force